# Task 3 CNN Experiment: CIFAR-10 Multi-Class Classification

Goal: run the convolutional-network experiment for all 10 CIFAR-10 classes in a separate notebook from the fully connected MLP sweep.

This notebook keeps image tensors in `3 x 32 x 32` form so the model can learn local spatial features such as edges, textures, and object parts. The MLP sweep remains in `HW4_p1_task_3_MCC_mlp_cnn.ipynb`.


## Direct Answers

- CNN input shape: `3 x 32 x 32` for RGB CIFAR-10 images.
- Output neurons: `10`, one logit for each CIFAR-10 class.
- Last-layer activation: no explicit activation when training with `torch.nn.CrossEntropyLoss`; use raw logits during training and `softmax` only when interpreting probabilities after inference.


In [ ]:
import os
from pathlib import Path
import random

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _base / "src"
    if (_src_dir / "notebook_progress.py").exists():
        if str(_src_dir) not in sys.path:
            sys.path.insert(0, str(_src_dir))
        break

from notebook_progress import tqdm

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer Apple Silicon GPU on macOS, then CUDA, then CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


In [ ]:
# Use the same CIFAR-10 files if they already exist under the project data directory.
def find_project_data_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        data_dir = base / "data"
        if data_dir.exists():
            return data_dir
    return Path.cwd() / "data"

DATA_DIR = find_project_data_dir()
candidate_roots = [DATA_DIR, Path("../data"), Path("src/data"), Path("data")]
data_root = next(
    (root for root in candidate_roots if (root / "cifar-10-batches-py").exists() or (root / "cifar-10-python.tar.gz").exists()),
    DATA_DIR,
)

train_dataset_raw = torchvision.datasets.CIFAR10(root=str(data_root), train=True, download=True)
test_dataset_raw = torchvision.datasets.CIFAR10(root=str(data_root), train=False, download=True)

class_names = train_dataset_raw.classes
train_images = train_dataset_raw.data
test_images = test_dataset_raw.data
train_labels = np.array(train_dataset_raw.targets)
test_labels = np.array(test_dataset_raw.targets)

len(train_images), len(test_images), class_names


In [ ]:
def compute_rgb_stats(images):
    pixels = images.astype(np.float32) / 255.0
    mean = pixels.mean(axis=(0, 1, 2), keepdims=True)
    std = pixels.std(axis=(0, 1, 2), keepdims=True) + 1e-7
    return mean, std


def make_optimizer(model, optimizer_name, lr, weight_decay=0.0):
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if optimizer_name == "sgd_momentum":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")


def evaluate(model, loader, criterion):
    model.eval()
    losses = []
    preds = []
    targets = []

    with torch.no_grad():
        for x_batch, y_batch in tqdm(loader, desc="Evaluating", leave=False):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item())
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            targets.extend(y_batch.cpu().numpy().tolist())

    accuracy = np.mean(np.array(preds) == np.array(targets))
    return float(np.mean(losses)), float(accuracy)


## CNN Extension

The MLPs above flatten each image into a vector. The CNN below keeps the image layout as channels, height, and width so the model can learn local features such as edges, textures, and object parts.


In [ ]:
class CIFAR10ImageDataset(Dataset):
    def __init__(self, images, labels, preprocessing="standardize", stats=None, augment=False):
        self.images = images
        self.labels = np.asarray(labels)
        self.preprocessing = preprocessing
        self.stats = stats
        self.augment = augment

        if preprocessing not in {"scale", "standardize"}:
            raise ValueError(f"CNN preprocessing must be 'scale' or 'standardize', got: {preprocessing}")
        if preprocessing == "standardize" and stats is None:
            raise ValueError("standardize preprocessing requires train-set stats")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx].astype(np.float32) / 255.0

        if self.augment and random.random() < 0.5:
            x = np.ascontiguousarray(np.flip(x, axis=1))

        if self.preprocessing == "standardize":
            mean, std = self.stats
            x = (x - mean.reshape(1, 1, 3)) / std.reshape(1, 1, 3)

        # CNNs expect channel-first image tensors: C x H x W.
        x = np.transpose(x, (2, 0, 1)).copy()
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_cnn_loaders(preprocessing="standardize", batch_size=128, augment=True):
    stats = compute_rgb_stats(train_images) if preprocessing == "standardize" else None
    train_ds = CIFAR10ImageDataset(
        train_images,
        train_labels,
        preprocessing=preprocessing,
        stats=stats,
        augment=augment,
    )
    test_ds = CIFAR10ImageDataset(
        test_images,
        test_labels,
        preprocessing=preprocessing,
        stats=stats,
        augment=False,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


class SmallCIFAR10CNN(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout / 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout / 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def clone_state_dict_to_cpu(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def train_one_cnn_experiment(config):
    train_loader, test_loader = make_cnn_loaders(
        preprocessing=config["preprocessing"],
        batch_size=config["batch_size"],
        augment=config.get("augment", True),
    )

    model = SmallCIFAR10CNN(dropout=config.get("dropout", 0.25)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(
        model,
        optimizer_name=config["optimizer"],
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 0.0),
    )

    scheduler = None
    if config.get("scheduler") == "reduce_on_plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=config.get("scheduler_factor", 0.5),
            patience=config.get("scheduler_patience", 5),
            min_lr=config.get("min_lr", 1e-5),
        )

    history = []
    best_accuracy = -1.0
    best_epoch = None
    best_test_loss = None
    best_state_dict = None
    best_loss_for_stop = float("inf")
    epochs_without_loss_improvement = 0
    stopped_early = False
    min_delta = config.get("early_stopping_min_delta", 1e-4)

    epoch_bar = tqdm(range(1, config["epochs"] + 1), desc=f"{config['name']} epochs", leave=True)
    for epoch in epoch_bar:
        model.train()
        train_losses = []
        lr_before_epoch = optimizer.param_groups[0]["lr"]

        batch_bar = tqdm(train_loader, desc=f"{config['name']} epoch {epoch}", leave=False)
        for x_batch, y_batch in batch_bar:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            batch_bar.set_postfix(loss=f"{loss.item():.4f}")

        test_loss, test_accuracy = evaluate(model, test_loader, criterion)

        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_epoch = epoch
            best_test_loss = test_loss
            best_state_dict = clone_state_dict_to_cpu(model)

        if test_loss < best_loss_for_stop - min_delta:
            best_loss_for_stop = test_loss
            epochs_without_loss_improvement = 0
        else:
            epochs_without_loss_improvement += 1

        if scheduler is not None:
            scheduler.step(test_loss)

        lr_after_scheduler = optimizer.param_groups[0]["lr"]
        row = {
            "experiment": config["name"],
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
            "lr": lr_before_epoch,
            "next_lr": lr_after_scheduler,
            "epochs_without_loss_improvement": epochs_without_loss_improvement,
        }
        history.append(row)

        epoch_bar.set_postfix(
            train_loss=f"{row['train_loss']:.4f}",
            test_loss=f"{test_loss:.4f}",
            test_acc=f"{test_accuracy:.4f}",
            lr=f"{lr_after_scheduler:.2e}",
        )

        if (
            config.get("early_stopping", False)
            and epochs_without_loss_improvement >= config.get("early_stopping_patience", 12)
        ):
            stopped_early = True
            print(
                f"Early stopping at epoch {epoch}: test loss did not improve by "
                f"at least {min_delta} for {epochs_without_loss_improvement} epochs."
            )
            break

    if best_state_dict is not None:
        model.load_state_dict({name: value.to(device) for name, value in best_state_dict.items()})

    return {
        "name": config["name"],
        "config": config,
        "history": history,
        "best_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "best_test_loss": best_test_loss,
        "best_state_dict": best_state_dict,
        "final_epoch": history[-1]["epoch"] if history else None,
        "stopped_early": stopped_early,
    }


In [ ]:
cnn_configs = [
    {
        "name": "small_cnn_rgb_standardized_adam_60ep_plateau",
        "model_family": "CNN",
        "preprocessing": "standardize",
        "optimizer": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 128,
        "epochs": 60,
        "dropout": 0.25,
        "augment": True,
        "scheduler": "reduce_on_plateau",
        "scheduler_factor": 0.5,
        "scheduler_patience": 5,
        "min_lr": 1e-5,
        "early_stopping": True,
        "early_stopping_patience": 12,
        "early_stopping_min_delta": 1e-4,
    },
]

# Run after the MLP sweep to add a CNN row to the comparison.
RUN_CNN_SWEEP = True

cnn_results = []
if RUN_CNN_SWEEP:
    for config in tqdm(cnn_configs, desc="CNN experiments", leave=True):
        cnn_results.append(train_one_cnn_experiment(config))
else:
    print("Set RUN_CNN_SWEEP = True to run the CNN experiment.")


In [ ]:
if cnn_results:
    cnn_summary = pd.DataFrame([
        {
            "experiment": result["name"],
            "best_accuracy": result["best_accuracy"],
            "best_epoch": result["best_epoch"],
            "best_test_loss": result["best_test_loss"],
            "preprocessing": result["config"]["preprocessing"],
            "optimizer": result["config"]["optimizer"],
            "lr": result["config"]["lr"],
            "batch_size": result["config"]["batch_size"],
            "epochs_requested": result["config"].get("epochs"),
            "final_epoch": result.get("final_epoch"),
            "stopped_early": result.get("stopped_early", False),
            "scheduler": result["config"].get("scheduler", "none"),
        }
        for result in cnn_results
    ]).sort_values("best_accuracy", ascending=False)

    display(cnn_summary)
else:
    cnn_summary = pd.DataFrame()
    print("Run the CNN sweep to build the CNN summary table.")


In [ ]:
if cnn_results:
    best_cnn = max(cnn_results, key=lambda result: result["best_accuracy"])
    hist = pd.DataFrame(best_cnn["history"])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = [
        ("train_loss", "Training loss"),
        ("test_loss", "Test loss"),
        ("test_accuracy", "Test accuracy"),
    ]

    for ax, (metric, title) in zip(axes, metrics):
        ax.plot(hist["epoch"], hist[metric], marker="o", markersize=4)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(True)

    axes[0].set_ylabel("Loss")
    axes[1].set_ylabel("Loss")
    axes[2].set_ylabel("Accuracy")
    axes[2].set_ylim(0, 1)
    fig.suptitle(f"CNN Run: {best_cnn['name']}")
    fig.tight_layout()
    plt.show()
else:
    print("Run the CNN sweep before drawing CNN curves.")


In [ ]:
if cnn_results:
    best_cnn = max(cnn_results, key=lambda result: result["best_accuracy"])
    print(
        f"Best CNN: {best_cnn['name']} at epoch {best_cnn['best_epoch']} "
        f"with test accuracy {best_cnn['best_accuracy']:.4f} "
        f"and test loss {best_cnn['best_test_loss']:.4f}. "
        f"Training ended at epoch {best_cnn.get('final_epoch')} "
        f"(stopped_early={best_cnn.get('stopped_early', False)})."
    )
else:
    print("CNN experiment has not been run yet. Run the CNN cell above to get exact CNN conclusions.")


## Findings and Conclusions

Input/output and last-layer design:

- The CNN receives CIFAR-10 images as channel-first tensors shaped `3 x 32 x 32`, preserving spatial structure instead of flattening pixels into one vector.
- The model has `10` output neurons, one raw logit for each CIFAR-10 class.
- The final layer should return raw logits, with no `Softmax`, when using `nn.CrossEntropyLoss`. This loss applies `LogSoftmax` internally, so adding `Softmax` before it is unnecessary and can make optimization numerically worse.

CNN run results:

- The CNN configuration is `small_cnn_rgb_standardized_adam_60ep_plateau`: a small BatchNorm CNN with RGB standardization, horizontal-flip augmentation, Adam, raw logits for `nn.CrossEntropyLoss`, up to `60` epochs, `ReduceLROnPlateau`, best-checkpoint tracking, and early stopping.
- The 60-epoch CNN run completed all `60` epochs; early stopping did not trigger.
- Best CNN accuracy was reached at epoch `59`: train loss `0.4789`, test loss `0.4516`, and test accuracy `0.8492`.
- Best CNN test loss was reached at epoch `57`: train loss `0.4851`, test loss `0.4507`, and test accuracy `0.8464`.
- The final epoch was slightly worse than the best checkpoint by accuracy: epoch `60` had train loss `0.4827`, test loss `0.4563`, and test accuracy `0.8437`.
- The scheduler did not reduce the learning rate during this run: LR stayed at `0.001` from epoch `1` through epoch `60`, because test loss kept improving often enough to avoid the plateau condition.

Overall conclusion:

The CNN is a more natural architecture for CIFAR-10 because it keeps color channels and local spatial structure. The best CNN checkpoint reached `0.8492` test accuracy at epoch `59`, far above the MLP task target of `0.53`. Report the best checkpoint by accuracy, not the final epoch, because epoch `60` was slightly worse than epoch `59`.
